In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/testing-model/Twice-Mixing-model.pth
/kaggle/input/image-underwater-11/eval/high-quality/2414_CLAHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2458_GLCHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2317_GLCHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2426_RED.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2563_GLCHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2326_GLCHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2481_CLAHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2493_GLCHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2402_DCP.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2581_Fusion.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2557_Histogram_prior.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2494_DCP.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2339_CLAHE.jpg
/kaggle/input/image-underwater-11/eval/high-quality/2404_Fusion.jpg
/kagg

In [2]:
# Necessary libraries and deterministic seed setup
import os
import random
import time
import math
from typing import Tuple, List

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import csv

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from scipy.stats import spearmanr, kendalltau

# Deterministic seed (documented for reproducibility)
SEED = 0
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Torch deterministic flags
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Deterministic seed set to {SEED}. torch.__version__ = {torch.__version__}")


Deterministic seed set to 0. torch.__version__ = 2.6.0+cu124


In [3]:
# Edit these paths / params as required
DATA_ROOT = "/kaggle/input/image-underwater-11"   # <- your dataset root
SAVE_DIR  = "runs/twice_mix_exp"
DEVICE    = "cuda:0" if torch.cuda.is_available() else "cpu"

# Training hyperparameters (paper exact)
LR = 1e-6
BATCH_SIZE = 1
EPOCHS = 20             # change as needed
MARGIN_EPS = 0.5        # epsilon (paper)
MIN_K_DIFF = 0.1
KS_TEST = [0.0, 0.2, 0.4, 0.6, 0.8]

os.makedirs(SAVE_DIR, exist_ok=True)
print("DEVICE:", DEVICE)


DEVICE: cuda:0


In [4]:
# Mapping by numeric ID + MappedTwiceMixDataset
import os, re, csv, shutil, pprint
from collections import defaultdict
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T

DATA_ROOT = "/kaggle/input/image-underwater-11"   # parent that contains 'train'
SPLIT = "train"
BASE_SPLIT = os.path.join(DATA_ROOT, SPLIT)
print("BASE_SPLIT:", BASE_SPLIT)
print("Subfolders:", sorted(os.listdir(BASE_SPLIT)))

orig_dir = os.path.join(BASE_SPLIT, "original")
hq_dir   = os.path.join(BASE_SPLIT, "high-quality")
lq_dir   = os.path.join(BASE_SPLIT, "low-quality")

# 1) helper: extract first numeric ID from filename
def extract_numeric_id(fname):
    """
    Return the first contiguous digit sequence in the filename (without extension).
    E.g., '1174.jpg' -> '1174', '1174_GC.jpg' -> '1174'.
    If none found, return None.
    """
    stem = os.path.splitext(fname)[0]
    m = re.search(r'(\d+)', stem)
    return m.group(1) if m else None

# 2) build maps id -> filenames
def build_id_map(folder):
    files = sorted([f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder,f))])
    id_map = defaultdict(list)
    for f in files:
        id_ = extract_numeric_id(f)
        if id_ is None:
            # optionally map by full stem fallback
            id_ = os.path.splitext(f)[0].lower()
        id_map[id_].append(f)
    return id_map

orig_map = build_id_map(orig_dir)
hq_map   = build_id_map(hq_dir)
lq_map   = build_id_map(lq_dir)

print("Counts by ID:")
print("  unique orig IDs:", len(orig_map))
print("  unique hq IDs:  ", len(hq_map))
print("  unique lq IDs:  ", len(lq_map))

# 3) find IDs present in all three
common_ids = sorted([id_ for id_ in orig_map.keys() if id_ in hq_map and id_ in lq_map])
print("IDs present in ALL three:", len(common_ids))

# 4) for each common id choose filenames to map.
# If multiple files exist for a given id in a folder, try to pick the file whose stem equals the id or closest.
def pick_best_for_id(id_, filelist):
    # prefer exact 'id' match (like '1174.jpg' or '1174.png')
    for f in filelist:
        stem = os.path.splitext(f)[0]
        if stem == id_:
            return f
    # else prefer files starting with 'id' + '_' (like '1174_GC.jpg')
    for f in filelist:
        if os.path.splitext(f)[0].lower().startswith(id_ + "_"):
            return f
    # else return first
    return filelist[0]

mappings = []
for id_ in common_ids:
    orig_f = pick_best_for_id(id_, orig_map[id_])
    hq_f   = pick_best_for_id(id_, hq_map[id_])
    lq_f   = pick_best_for_id(id_, lq_map[id_])
    mappings.append((orig_f, hq_f, lq_f))

# 5) diagnostics and sample
print("Total mapped triplets (by numeric ID):", len(mappings))
print("Sample mappings (first 20):")
pprint.pprint(mappings[:20])

# 6) Save mapping CSV for training
out_csv_path = "/kaggle/working/mapping_train_triplets.csv"
with open(out_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["orig_filename","hq_filename","lq_filename"])
    for orig_f, hq_f, lq_f in mappings:
        writer.writerow([orig_f, hq_f, lq_f])
print("Saved mapping CSV to:", out_csv_path)

# 7) Optionally create aligned copies (or symlinks) under /kaggle/working/aligned_train/
MAKE_ALIGNED_COPY = True
ALIGNED_ROOT = "/kaggle/working/aligned_train"
if MAKE_ALIGNED_COPY:
    aligned_orig = os.path.join(ALIGNED_ROOT, "original"); os.makedirs(aligned_orig, exist_ok=True)
    aligned_hq   = os.path.join(ALIGNED_ROOT, "high_quality"); os.makedirs(aligned_hq, exist_ok=True)
    aligned_lq   = os.path.join(ALIGNED_ROOT, "low_quality"); os.makedirs(aligned_lq, exist_ok=True)
    # copy or symlink
    for orig_f, hq_f, lq_f in mappings:
        src_o = os.path.join(orig_dir, orig_f)
        src_h = os.path.join(hq_dir, hq_f)
        src_l = os.path.join(lq_dir, lq_f)
        dst_o = os.path.join(aligned_orig, orig_f)
        dst_h = os.path.join(aligned_hq, hq_f)
        dst_l = os.path.join(aligned_lq, lq_f)
        # prefer symlink when possible for speed, else copy
        try:
            if not os.path.exists(dst_o):
                os.symlink(src_o, dst_o)
            if not os.path.exists(dst_h):
                os.symlink(src_h, dst_h)
            if not os.path.exists(dst_l):
                os.symlink(src_l, dst_l)
        except Exception as e:
            # fallback to copying (safe)
            shutil.copy(src_o, dst_o)
            shutil.copy(src_h, dst_h)
            shutil.copy(src_l, dst_l)
    print("Aligned copies/symlinks created at:", ALIGNED_ROOT)

# 8) Provide a Mapped dataset class that reads the CSV and uses full paths
class MappedTwiceMixDataset(Dataset):
    """
    Dataset that takes:
      - base_split_dir: e.g. '/kaggle/input/image-underwater-11/train'
      - mapping_csv: rows of orig_filename,hq_filename,lq_filename (filenames relative to respective folders)
    Returns (orig_tensor, hq_tensor, lq_tensor, orig_filename)
    """
    def __init__(self, base_split_dir, mapping_csv, transform=None, resize=(224,224), folders=None):
        self.base = base_split_dir
        # folders mapping default
        if folders is None:
            folders = {"orig":"original","hq":"high-quality","lq":"low-quality"}
        self.folders = folders
        # load mapping_csv
        rows = []
        with open(mapping_csv, "r") as f:
            r = csv.reader(f)
            header = next(r)
            for row in r:
                if len(row) >= 3:
                    rows.append(tuple(row[:3]))
        self.mappings = rows
        if transform is None:
            self.transform = T.Compose([
                T.Resize(resize),
                T.ToTensor(),
                T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])
        else:
            self.transform = transform

    def __len__(self):
        return len(self.mappings)

    def __getitem__(self, idx):
        orig_f, hq_f, lq_f = self.mappings[idx]
        p_orig = os.path.join(self.base, self.folders["orig"], orig_f)
        p_hq   = os.path.join(self.base, self.folders["hq"],   hq_f)
        p_lq   = os.path.join(self.base, self.folders["lq"],   lq_f)
        img_orig = Image.open(p_orig).convert('RGB')
        img_hq   = Image.open(p_h).convert('RGB') if False else Image.open(p_hq).convert('RGB')
        img_lq   = Image.open(p_lq).convert('RGB')
        if self.transform:
            img_orig = self.transform(img_orig)
            img_hq   = self.transform(img_hq)
            img_lq   = self.transform(img_lq)
        return img_orig, img_hq, img_lq, orig_f

# 9) Quick smoke instantiate the mapped dataset (either from mapping CSV or the aligned folder)
mapped_csv = out_csv_path
ds_mapped = MappedTwiceMixDataset(BASE_SPLIT, mapped_csv)
print("Mapped dataset length:", len(ds_mapped))
if len(ds_mapped) > 0:
    o,h,l,f = ds_mapped[0]
    print("Sample shapes:", o.shape, h.shape, l.shape, "filename:", f)

# Done. Now you can replace previous dataset usage with `ds_mapped` or point training to ALIGNED_ROOT (if you created copies).


BASE_SPLIT: /kaggle/input/image-underwater-11/train
Subfolders: ['high-quality', 'low-quality', 'original']
Counts by ID:
  unique orig IDs: 2000
  unique hq IDs:   2000
  unique lq IDs:   1597
IDs present in ALL three: 1597
Total mapped triplets (by numeric ID): 1597
Sample mappings (first 20):
[('100.jpg', '100_GLCHE.jpg', '100_all_in_one.jpg'),
 ('1003.jpg', '1003_GC.jpg', '1003_ULAP.jpg'),
 ('1004.jpg', '1004_CLAHE.jpg', '1004_ULAP.jpg'),
 ('1007.jpg', '1007_Fusion.jpg', '1007_all_in_one.jpg'),
 ('1008.jpg', '1008_GLCHE.jpg', '1008_ULAP.jpg'),
 ('1011.jpg', '1011_Fusion.jpg', '1011_Retinex.jpg'),
 ('1012.jpg', '1012_CLAHE.jpg', '1012_all_in_one.jpg'),
 ('1013.jpg', '1013_CLAHE.jpg', '1013_Retinex.jpg'),
 ('1014.jpg', '1014_GLCHE.jpg', '1014_all_in_one.jpg'),
 ('1015.jpg', '1015_Water-net.jpg', '1015_HE.jpg'),
 ('1016.jpg', '1016_Histogram_prior.jpg', '1016_Retinex.jpg'),
 ('1017.jpg', '1017_CLAHE.jpg', '1017_Retinex.jpg'),
 ('1019.jpg', '1019_GC.jpg', '1019_ULAP.jpg'),
 ('102.jpg',

In [5]:
# utils: sampling K1,K2, mixing, metrics computation and unnormalize helper
def sample_Ks(min_diff: float = 0.1, sampler: str = "uniform", beta_alpha: float = 0.5) -> Tuple[float,float]:
    """
    Sample two Ks in (0,1) such that |K1-K2| >= min_diff.
    sampler: 'uniform' or 'beta' (ablation)
    """
    while True:
        if sampler == "uniform":
            k1, k2 = random.random(), random.random()
        elif sampler == "beta":
            k1 = float(np.random.beta(beta_alpha, beta_alpha))
            k2 = float(np.random.beta(beta_alpha, beta_alpha))
        else:
            raise ValueError("sampler must be 'uniform' or 'beta'")
        if abs(k1 - k2) >= min_diff:
            return float(k1), float(k2)

def mix_images(hq: torch.Tensor, lq: torch.Tensor, k: float) -> torch.Tensor:
    """
    k * HQ + (1-k) * LQ
    hq/lq are tensors (B,C,H,W) or (C,H,W)
    """
    return k * hq + (1.0 - k) * lq

def compute_srcc_krcc_list(preds: List[float], gt: List[float]) -> Tuple[float,float]:
    """
    Spearman and Kendall between preds and gt lists.
    """
    if len(preds) < 2:
        return 0.0, 0.0
    sr, _ = spearmanr(preds, gt)
    kt, _ = kendalltau(preds, gt)
    if np.isnan(sr): sr = 0.0
    if np.isnan(kt): kt = 0.0
    return float(sr), float(kt)

def aggregate_scores_matrix(scores_matrix: np.ndarray, ks: List[float]):
    """
    scores_matrix: shape (N_images, len(ks))
    For each image, compute SRCC & KRCC between the predicted scores and ground truth (which is the ranking by ks)
    Return mean & std for SRCC and KRCC across images.
    """
    sr_list, kr_list = [], []
    for row in scores_matrix:
        sr, kr = compute_srcc_krcc_list(row.tolist(), ks)
        sr_list.append(sr)
        kr_list.append(kr)
    return float(np.mean(sr_list)), float(np.std(sr_list)), float(np.mean(kr_list)), float(np.std(kr_list))

# helper for unnormalize (for visualization)
def unnormalize_tensor(t: torch.Tensor):
    # t: (C,H,W), normalized by ImageNet mean/std
    mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
    std  = torch.tensor([0.229,0.224,0.225])[:,None,None]
    return (t * std + mean).clamp(0,1)


In [6]:
class VGGRanker(nn.Module):
    """
    VGG16 conv features -> GlobalAvgPool -> 3 FC layers -> scalar output
    Default FC sizes: 512 -> 128 -> 32 -> 1
    """
    def __init__(self, pretrained=False, fc_sizes=(512,128,32,1)):
        super().__init__()
        vgg = models.vgg16(pretrained=pretrained)
        # conv features
        self.features = vgg.features
        # GAP to (B,512,1,1) -> flatten to (B,512)
        self.gap = nn.AdaptiveAvgPool2d(1)
        # build FC layers
        layers = []
        in_ch = fc_sizes[0]
        for out_ch in fc_sizes[1:]:
            layers.append(nn.Linear(in_ch, out_ch))
            if out_ch != 1:
                layers.append(nn.ReLU(inplace=True))
            in_ch = out_ch
        self.fc = nn.Sequential(*layers)

    def forward(self, x):
        feat = self.features(x)               # (B,512,H',W')
        pooled = self.gap(feat).view(x.size(0), -1)  # (B,512)
        score = self.fc(pooled)               # (B,1)
        return score

# Quick smoke (optional)
_device = "cuda:0" if torch.cuda.is_available() else "cpu"
_model = VGGRanker(pretrained=False).to(_device)
_dummy = torch.randn(1,3,224,224).to(_device)
print("VGGRanker output shape:", _model(_dummy).shape)
del _model, _dummy


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


VGGRanker output shape: torch.Size([1, 1])


In [7]:
import torch

MARGIN_EPS = 0.5  # epsilon exactly as the paper requires

def paper_margin_loss(s1: torch.Tensor, s2: torch.Tensor, gamma: float, eps: float = MARGIN_EPS) -> torch.Tensor:
    """
    Compute paper loss L = max(0, (s1 - s2) * gamma + eps)
    s1, s2 can be (B,1) or (B,)
    gamma = 1 if K1 < K2 else -1 (per paper)
    Returns mean over batch.
    """
    s1 = s1.view(-1)
    s2 = s2.view(-1)
    vals = (s1 - s2) * gamma + eps
    loss = torch.clamp(vals, min=0.0)
    return loss.mean()


In [8]:
import os, time
import torch.optim as optim
from torch.utils.data import DataLoader
import csv

def train_one_epoch(model, dataloader, optimizer, device, epoch, sampler="uniform", min_diff=0.1):
    model.train()
    total_loss = 0.0
    n = 0
    start = time.time()
    for i, (orig, hq, lq, fname) in enumerate(dataloader):
        # shapes: (1,C,H,W)
        hq = hq.to(device); lq = lq.to(device)
        # sample Ks and create mixes
        k1, k2 = sample_Ks(min_diff=min_diff, sampler=sampler)
        mix1 = mix_images(hq, lq, k1)
        mix2 = mix_images(hq, lq, k2)
        # forward
        s1 = model(mix1)   # (1,1)
        s2 = model(mix2)
        # gamma per paper
        gamma = 1.0 if k1 < k2 else -1.0
        loss = paper_margin_loss(s1, s2, gamma, eps=MARGIN_EPS)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
        if n % 200 == 0:
            print(f"[Epoch {epoch}] Step {n} Loss {loss.item():.6f} k1={k1:.3f} k2={k2:.3f} gamma={gamma}")
    elapsed = time.time() - start
    avg_loss = total_loss / max(1, n)
    print(f"Epoch {epoch} finished - avg_loss: {avg_loss:.6f} - time: {elapsed:.1f}s")
    return avg_loss

def train_full(root_split_dir, mapping_csv, save_dir, device, epochs=10, lr=1e-6, pretrained=False, sampler="uniform"):
    set_seed(SEED)
    # dataset that reads mapping CSV
    train_ds = MappedTwiceMixDataset(root_split_dir, mapping_csv)
    loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
    model = VGGRanker(pretrained=pretrained).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []
    os.makedirs(save_dir, exist_ok=True)
    for e in range(1, epochs+1):
        avg_loss = train_one_epoch(model, loader, optimizer, device, e, sampler=sampler)
        history.append({"epoch": e, "avg_loss": avg_loss})
        # checkpoint
        ckpt_path = os.path.join(save_dir, f"ckpt_epoch{e}.pt")
        torch.save({"epoch": e, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict()}, ckpt_path)
        print("Saved checkpoint:", ckpt_path)
        # append CSV
        hist_csv = os.path.join(save_dir, "train_history.csv")
        with open(hist_csv, "w", newline="") as f:
            w = csv.writer(f); w.writerow(["epoch","avg_loss"])
            for r in history: w.writerow([r["epoch"], r["avg_loss"]])
    return model, history

# Example to run (smoke): (uncomment to run)
# model, history = train_full("/kaggle/input/image-underwater-11/train", "/kaggle/working/mapping_train_triplets.csv", "/kaggle/working/twice_mix_runs", DEVICE, epochs=1, lr=1e-6)


In [9]:

def evaluate_synthetic(model, root_split_dir, mapping_csv, device, ks=[0.0,0.2,0.4,0.6,0.8], batch_size=1):
    model.eval()
    ds = MappedTwiceMixDataset(root_split_dir, mapping_csv)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    scores = []
    filenames = []
    with torch.no_grad():
        for orig, hq, lq, fname in loader:
            hq = hq.to(device); lq = lq.to(device)
            row = []
            for k in ks:
                mix = mix_images(hq, lq, k)
                s = model(mix).cpu().numpy().ravel()[0]
                row.append(float(s))
            scores.append(row)
            filenames.append(fname[0])
    scores = np.array(scores)  # N x len(ks)
    sr_mean, sr_std, kr_mean, kr_std = aggregate_scores_matrix(scores, ks)
    print("Synthetic evaluation results:")
    print(f"SRCC mean: {sr_mean:.4f}, std: {sr_std:.4f}")
    print(f"KRCC mean: {kr_mean:.4f}, std: {kr_std:.4f}")
    # save per-image CSV
    out_csv = os.path.join("/kaggle/working", "synthetic_test_scores.csv")
    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["filename"] + [f"K={k}" for k in ks])
        for fn, row in zip(filenames, scores.tolist()):
            w.writerow([fn] + row)
    print("Saved synthetic per-image scores to:", out_csv)
    return {"sr_mean": sr_mean, "sr_std": sr_std, "kr_mean": kr_mean, "kr_std": kr_std}


In [10]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({"epoch": epoch, "model_state": model.state_dict(), "opt_state": optimizer.state_dict()}, path)
    print("Saved checkpoint:", path)

def load_checkpoint(model, ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print("Loaded checkpoint epoch:", ckpt.get("epoch", "unknown"))
    return model


In [11]:
# quick inline tests
def unit_tests_quick():
    # test sample_Ks
    for _ in range(50):
        k1,k2 = sample_Ks(min_diff=0.1)
        assert abs(k1 - k2) >= 0.1
    # test mix_images linearity
    hq = torch.ones(1,3,4,4) * 2.0
    lq = torch.zeros(1,3,4,4)
    m = mix_images(hq, lq, 0.25)
    assert torch.allclose(m, torch.ones_like(m) * 0.5)
    # test paper loss numerics
    s1 = torch.tensor([0.0]); s2 = torch.tensor([1.0]); gamma = 1.0
    assert abs(paper_margin_loss(s1,s2,gamma).item() - 0.0) < 1e-6
    s1b = torch.tensor([1.0]); s2b = torch.tensor([0.0])
    assert abs(paper_margin_loss(s1b,s2b,gamma).item() - 1.5) < 1e-6
    print("All quick unit tests passed.")

unit_tests_quick()


All quick unit tests passed.


In [12]:
# SMOKE TRAIN (1 epoch)
save_dir = "/kaggle/working/twice_mix_run_smoke"
os.makedirs(save_dir, exist_ok=True)

# train_full defined earlier. If not loaded, copy the function cell for train_full
model, history = train_full(
    root_split_dir="/kaggle/input/image-underwater-11/train",
    mapping_csv="/kaggle/working/mapping_train_triplets.csv",
    save_dir=save_dir,
    device=DEVICE,
    epochs=1,         # smoke: 1 epoch
    lr=1e-6,
    pretrained=False, # change to True only if you allow pretrained weights download
    sampler="uniform" # default; change to "beta" for ablation
)
print("Smoke training done. Checkpoint(s) in:", save_dir)


[Epoch 1] Step 200 Loss 0.499948 k1=0.426 k2=0.566 gamma=1.0
[Epoch 1] Step 400 Loss 0.414618 k1=0.925 k2=0.586 gamma=-1.0
[Epoch 1] Step 600 Loss 0.361106 k1=0.645 k2=0.473 gamma=-1.0
[Epoch 1] Step 800 Loss 0.288906 k1=0.134 k2=0.627 gamma=1.0
[Epoch 1] Step 1000 Loss 0.551952 k1=0.240 k2=0.018 gamma=-1.0
[Epoch 1] Step 1200 Loss 0.520213 k1=0.637 k2=0.901 gamma=1.0
[Epoch 1] Step 1400 Loss 0.243509 k1=0.049 k2=0.188 gamma=1.0
Epoch 1 finished - avg_loss: 0.329262 - time: 46.7s
Saved checkpoint: /kaggle/working/twice_mix_run_smoke/ckpt_epoch1.pt
Smoke training done. Checkpoint(s) in: /kaggle/working/twice_mix_run_smoke


In [13]:
# LOAD CHECKPOINT AND EVALUATE
ckpt_path = "/kaggle/working/twice_mix_run_smoke/ckpt_epoch1.pt"
model = VGGRanker(pretrained=False).to(DEVICE)
model = load_checkpoint(model, ckpt_path, DEVICE)

res = evaluate_synthetic(
    model=model,
    root_split_dir="/kaggle/input/image-underwater-11/train",            # or test path when available
    mapping_csv="/kaggle/working/mapping_train_triplets.csv",
    device=DEVICE,
    ks=[0.0, 0.2, 0.4, 0.6, 0.8],
    batch_size=1
)
print("Synthetic evaluation:", res)


Loaded checkpoint epoch: 1
Synthetic evaluation results:
SRCC mean: 0.6426, std: 0.7552
KRCC mean: 0.6433, std: 0.7519
Saved synthetic per-image scores to: /kaggle/working/synthetic_test_scores.csv
Synthetic evaluation: {'sr_mean': 0.6426424546023793, 'sr_std': 0.7551974155191137, 'kr_mean': 0.6433312460864118, 'kr_std': 0.7518549291894929}


In [14]:
def visualize_sample_from_mapped(ds, idx=0, ks=[0.0,0.25,0.5,0.75,1.0]):
    """
    Visualize original, HQ, LQ, and mixed images for a given dataset index.
    """
    # fetch one triplet
    orig, hq, lq, fname = ds[idx]
    
    # unnormalize helper
    def unnormalize_tensor(t: torch.Tensor):
        mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
        std  = torch.tensor([0.229,0.224,0.225])[:,None,None]
        return (t * std + mean).clamp(0,1)

    orig_u = unnormalize_tensor(orig)
    hq_u   = unnormalize_tensor(hq)
    lq_u   = unnormalize_tensor(lq)

    # show all images
    n = 3 + len(ks)
    plt.figure(figsize=(3*n, 4))

    # show original, HQ, LQ
    plt.subplot(1, n, 1)
    plt.imshow(orig_u.permute(1,2,0).cpu().numpy())
    plt.title("Original"); plt.axis("off")

    plt.subplot(1, n, 2)
    plt.imshow(hq_u.permute(1,2,0).cpu().numpy())
    plt.title("High Quality"); plt.axis("off")

    plt.subplot(1, n, 3)
    plt.imshow(lq_u.permute(1,2,0).cpu().numpy())
    plt.title("Low Quality"); plt.axis("off")

    # show mixed images
    for i, k in enumerate(ks):
        mix = k * hq + (1 - k) * lq
        mix_u = unnormalize_tensor(mix)
        plt.subplot(1, n, 4 + i)
        plt.imshow(mix_u.permute(1,2,0).cpu().numpy())
        plt.title(f"K={k:.2f}"); plt.axis("off")

    plt.suptitle(f"Sample: {fname}")
    plt.show()


In [ ]:
# VISUALIZE SAMPLE MIXES
ds_mapped = MappedTwiceMixDataset("/kaggle/input/image-underwater-11/train", "/kaggle/working/mapping_train_triplets.csv")
visualize_sample_from_mapped(ds_mapped, idx=0, ks=[0.0, 0.2, 0.4, 0.6, 0.8])
# try a few different indices (e.g., idx=10, idx=100)


In [16]:
# RESUME TRAINING FROM CHECKPOINT
ckpt_path = "/kaggle/working/twice_mix_run_smoke/ckpt_epoch1.pt"
model = VGGRanker(pretrained=False).to(DEVICE)
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)

start_epoch = ckpt.get("epoch", 1) + 1
# Continue training for N more epochs
N_more = 4
train_ds = MappedTwiceMixDataset("/kaggle/input/image-underwater-11/train", "/kaggle/working/mapping_train_triplets.csv")
loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
for e in range(start_epoch, start_epoch + N_more):
    avg_loss = train_one_epoch(model, loader, optimizer, DEVICE, e, sampler="uniform")
    # save each epoch
    torch.save({"epoch": e, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict()},
               f"/kaggle/working/twice_mix_run_smoke/ckpt_epoch{e}.pt")


[Epoch 2] Step 200 Loss 0.000000 k1=0.670 k2=0.936 gamma=1.0
[Epoch 2] Step 400 Loss 0.160237 k1=0.451 k2=0.851 gamma=1.0
[Epoch 2] Step 600 Loss 1.119639 k1=0.881 k2=0.060 gamma=-1.0
[Epoch 2] Step 800 Loss 0.480489 k1=0.956 k2=0.417 gamma=-1.0
[Epoch 2] Step 1000 Loss 0.052816 k1=0.582 k2=0.883 gamma=1.0
[Epoch 2] Step 1200 Loss 0.245731 k1=0.019 k2=0.251 gamma=1.0
[Epoch 2] Step 1400 Loss 0.000000 k1=0.385 k2=0.590 gamma=1.0
Epoch 2 finished - avg_loss: 0.224625 - time: 46.7s
[Epoch 3] Step 200 Loss 0.443351 k1=0.301 k2=0.467 gamma=1.0
[Epoch 3] Step 400 Loss 0.000000 k1=0.478 k2=0.143 gamma=-1.0
[Epoch 3] Step 600 Loss 0.407248 k1=0.021 k2=0.773 gamma=1.0
[Epoch 3] Step 800 Loss 0.000000 k1=0.807 k2=0.381 gamma=-1.0
[Epoch 3] Step 1000 Loss 0.000000 k1=0.367 k2=0.715 gamma=1.0
[Epoch 3] Step 1200 Loss 0.148840 k1=0.246 k2=0.405 gamma=1.0
[Epoch 3] Step 1400 Loss 0.000000 k1=0.393 k2=0.287 gamma=-1.0
Epoch 3 finished - avg_loss: 0.190572 - time: 46.7s
[Epoch 4] Step 200 Loss 0.32120

In [17]:
# FULL TRAIN (example)
save_dir_full = "/kaggle/working/twice_mix_run_full"
model, history = train_full(
    root_split_dir="/kaggle/input/image-underwater-11/train",
    mapping_csv="/kaggle/working/mapping_train_triplets.csv",
    save_dir=save_dir_full,
    device=DEVICE,
    epochs=60,         # choose 20/30/50 per compute budget
    lr=1e-6,
    pretrained=False,
    sampler="uniform"
)


[Epoch 1] Step 200 Loss 0.499948 k1=0.426 k2=0.566 gamma=1.0
[Epoch 1] Step 400 Loss 0.414618 k1=0.925 k2=0.586 gamma=-1.0
[Epoch 1] Step 600 Loss 0.361106 k1=0.645 k2=0.473 gamma=-1.0
[Epoch 1] Step 800 Loss 0.288906 k1=0.134 k2=0.627 gamma=1.0
[Epoch 1] Step 1000 Loss 0.551952 k1=0.240 k2=0.018 gamma=-1.0
[Epoch 1] Step 1200 Loss 0.520213 k1=0.637 k2=0.901 gamma=1.0
[Epoch 1] Step 1400 Loss 0.243509 k1=0.049 k2=0.188 gamma=1.0
Epoch 1 finished - avg_loss: 0.329262 - time: 46.7s
Saved checkpoint: /kaggle/working/twice_mix_run_full/ckpt_epoch1.pt
[Epoch 2] Step 200 Loss 1.510747 k1=0.670 k2=0.936 gamma=1.0
[Epoch 2] Step 400 Loss 0.904316 k1=0.451 k2=0.851 gamma=1.0
[Epoch 2] Step 600 Loss 0.000000 k1=0.881 k2=0.060 gamma=-1.0
[Epoch 2] Step 800 Loss 0.202483 k1=0.956 k2=0.417 gamma=-1.0
[Epoch 2] Step 1000 Loss 0.000000 k1=0.582 k2=0.883 gamma=1.0
[Epoch 2] Step 1200 Loss 0.547483 k1=0.019 k2=0.251 gamma=1.0
[Epoch 2] Step 1400 Loss 0.440072 k1=0.385 k2=0.590 gamma=1.0
Epoch 2 finishe

In [18]:
from pathlib import Path
import pandas as pd
import re

def create_triplet_mapping_by_id(root_dir: str, save_path: str):
    base = Path(root_dir)
    orig_dir = base / "original"
    hq_dir = base / "high-quality"
    lq_dir = base / "low-quality"

    # Collect jpg filenames
    orig_files = sorted([f.name for f in orig_dir.glob("*.jpg")])
    hq_files = sorted([f.name for f in hq_dir.glob("*.jpg")])
    lq_files = sorted([f.name for f in lq_dir.glob("*.jpg")])

    def get_id(name):
        return re.sub(r'[^0-9]', '', name)

    orig_ids = {get_id(f): f for f in orig_files}
    hq_ids   = {get_id(f): f for f in hq_files}
    lq_ids   = {get_id(f): f for f in lq_files}

    common = sorted(set(orig_ids) & set(hq_ids) & set(lq_ids))
    rows = [(orig_ids[c], hq_ids[c], lq_ids[c]) for c in common]

    df = pd.DataFrame(rows, columns=["original", "high_quality", "low_quality"])
    df.to_csv(save_path, index=False)
    print(f"✅ Saved mapping CSV: {save_path}")
    print(f"✅ Total triplets found: {len(df)}")

create_triplet_mapping_by_id(
    root_dir="/kaggle/input/image-underwater-11/eval",     # 👈 changed test → eval
    save_path="/kaggle/working/mapping_eval_triplets.csv"  # 👈 just renamed output file
)



✅ Saved mapping CSV: /kaggle/working/mapping_eval_triplets.csv
✅ Total triplets found: 218


In [19]:
import pandas as pd
df = pd.read_csv("/kaggle/working/mapping_eval_triplets.csv")
print("Rows in mapping_test_triplets.csv:", len(df))
df.head(10)


Rows in mapping_test_triplets.csv: 218


,original,high_quality,low_quality
0,2314.jpg,2314_Fusion.jpg,2314_ULAP.jpg
1,2315.jpg,2315_GLCHE.jpg,2315_all_in_one.jpg
2,2317.jpg,2317_GLCHE.jpg,2317_all_in_one.jpg
3,2319.jpg,2319_Fusion.jpg,2319_RED.jpg
4,2320.jpg,2320_GLCHE.jpg,2320_all_in_one.jpg
5,2321.jpg,2321_GLCHE.jpg,2321_DCP.jpg
6,2322.jpg,2322_Water-net.jpg,2322_ULAP.jpg
7,2323.jpg,2323_GLCHE.jpg,2323_all_in_one.jpg
8,2324.jpg,2324_CLAHE.jpg,2324_all_in_one.jpg
9,2326.jpg,2326_GLCHE.jpg,2326_GC.jpg


In [20]:
test_results = evaluate_synthetic(
    model=model,
    root_split_dir="/kaggle/input/image-underwater-11/eval",  # test (or eval) folder
    mapping_csv="/kaggle/working/mapping_eval_triplets.csv",  # mapping file you created
    device=DEVICE,
    ks=[0.0, 0.2, 0.4, 0.6, 0.8],  # K blending factors
    batch_size=1
)
print("Final evaluation results:", test_results)


Synthetic evaluation results:
SRCC mean: 0.9486, std: 0.2698
KRCC mean: 0.9431, std: 0.2737
Saved synthetic per-image scores to: /kaggle/working/synthetic_test_scores.csv
Final evaluation results: {'sr_mean': 0.9486238532110091, 'sr_std': 0.26976533427742955, 'kr_mean': 0.9431192660550457, 'kr_std': 0.2737329350307745}


In [21]:
import pandas as pd
df = pd.read_csv("/kaggle/working/synthetic_test_scores.csv")
print(df.head())


   filename     K=0.0     K=0.2     K=0.4     K=0.6     K=0.8
0  2314.jpg -2.479801 -0.266977  1.971523  4.072801  5.730646
1  2315.jpg -3.353301 -1.490729  0.175583  1.738166  3.032510
2  2317.jpg -3.484059 -1.910737  1.065974  3.307133  3.742778
3  2319.jpg  2.216410  3.861311  5.383505  6.737635  7.891346
4  2320.jpg -1.785662 -0.942743  0.130723  1.486820  2.821991


In [31]:
import torch
from PIL import Image
import torchvision.transforms as T
import os

# --- 1️⃣  Load model checkpoint ---
ckpt_path = "/kaggle/working/twice_mix_run_full/ckpt_epoch40.pt"  # 👈 change epoch if needed
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VGGRanker(pretrained=False).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Loaded model from {ckpt_path}")

# --- 2️⃣  Preprocessing same as training ---
transform = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

# --- 3️⃣  Define a function for predicting one image ---
def predict_quality(image_path):
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        score = model(x).item()
    return score

# --- 4️⃣  Example: predict single image quality ---
img_path = "/kaggle/input/image-underwater-11/eval/low-quality/2315_all_in_one.jpg"  # 👈 any image path
score = predict_quality(img_path)
print(f"Predicted quality score for {os.path.basename(img_path)}: {score:.4f}")


✅ Loaded model from /kaggle/working/twice_mix_run_full/ckpt_epoch40.pt
Predicted quality score for 2315_all_in_one.jpg: -2.4411


In [43]:
# --- 5️⃣  Predict and rank multiple images ---
image_list = [
    "/kaggle/input/image-underwater-11/eval/high-quality/2333_GLCHE.jpg",
    "/kaggle/input/image-underwater-11/eval/original/2342.jpg"
]

results = []
for img_path in image_list:
    score = predict_quality(img_path)
    results.append((img_path, score))

# Sort by score (higher = better)
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

print("\n📊 Quality ranking:")
for rank, (path, score) in enumerate(results_sorted, start=1):
    print(f"Rank {rank}: {os.path.basename(path)} | Score = {score:.4f}")



📊 Quality ranking:
Rank 1: 2333_GLCHE.jpg | Score = 5.8405
Rank 2: 2342.jpg | Score = 2.4367
